### Encoder-Decoder Model for Sequence-to-Sequence Prediction in pure PyTorch




In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from random import randint
from torch.utils.data import Dataset, DataLoader

In [2]:
# ****************** SECTION 1: DATA GENERATION FUNCTIONS ******************
def generate_sequence(length, n_unique):
    """Generate a sequence of random integers"""
    return [randint(1, n_unique-1) for _ in range(length)]

def one_hot_encode(sequence, n_unique):
    """One-hot encode a sequence"""
    encoding = np.zeros((len(sequence), n_unique), dtype=np.float32)
    for i, value in enumerate(sequence):
        encoding[i, value] = 1.0
    return encoding

def one_hot_decode(encoded_seq):
    """Decode a one-hot encoded sequence"""
    return [np.argmax(vector) for vector in encoded_seq]

In [3]:
# ****************** SECTION 2: DATASET AND DATALOADER ******************
class Seq2SeqDataset(Dataset):
    def __init__(self, n_in, n_out, cardinality, n_samples):
        self.n_in = n_in
        self.n_out = n_out
        self.cardinality = cardinality
        self.n_samples = n_samples
        self.data = self._generate_data()

    def _generate_data(self):
        data = []
        for _ in range(self.n_samples):
            # Generate source sequence
            source = generate_sequence(self.n_in, self.cardinality)

            # Define target sequence (reversed subset of source)
            target = source[:self.n_out]
            target.reverse()

            # Create padded input target sequence (for teacher forcing)
            target_in = [0] + target[:-1]

            # Store the raw sequences for easier debugging
            data.append({
                'source_raw': source,
                'target_raw': target,
                'target_in_raw': target_in,
                'source': one_hot_encode(source, self.cardinality),
                'target': one_hot_encode(target, self.cardinality),
                'target_in': one_hot_encode(target_in, self.cardinality)
            })
        return data

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'source': torch.FloatTensor(item['source']),
            'target': torch.FloatTensor(item['target']),
            'target_in': torch.FloatTensor(item['target_in']),
            'source_raw': item['source_raw'],
            'target_raw': item['target_raw']
        }

In [4]:
# ****************** SECTION 3: MODEL DEFINITION ******************
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1, dropout=0.0):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

    def forward(self, x):
        # x shape: [batch_size, seq_len, input_size]
        _, (hidden, cell) = self.lstm(x)
        return hidden, cell

class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, num_layers=1, dropout=0.0):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            output_size,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, hidden, cell):
        # x shape: [batch_size, 1, output_size]
        output, (hidden, cell) = self.lstm(x, (hidden, cell))

        # output shape: [batch_size, 1, hidden_size]
        output = self.softmax(self.out(output))

        # output shape: [batch_size, 1, output_size]
        return output, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, target_in, teacher_forcing_ratio=1.0):
        # source shape: [batch_size, source_len, input_size]
        # target_in shape: [batch_size, target_len, output_size]

        batch_size = source.size(0)
        target_len = target_in.size(1)
        target_size = target_in.size(2)

        # Initialize outputs tensor to store decoder outputs
        outputs = torch.zeros(batch_size, target_len, target_size).to(source.device)

        # Encode the source sequence
        hidden, cell = self.encoder(source)

        # First input to the decoder is the start token
        decoder_input = target_in[:, 0, :].unsqueeze(1)

        # Decode one step at a time
        for t in range(target_len):
            decoder_output, hidden, cell = self.decoder(decoder_input, hidden, cell)
            outputs[:, t, :] = decoder_output.squeeze(1)

            # For next iteration - use teacher forcing or previous prediction
            if t < target_len - 1:
                if torch.rand(1).item() < teacher_forcing_ratio:
                    decoder_input = target_in[:, t+1, :].unsqueeze(1)
                else:
                    decoder_input = decoder_output

        return outputs

In [5]:

#  ****************** SECTION 4: PREDICTION FUNCTION ******************
def predict_sequence(model, source, n_steps, device):
    """Generate target given source sequence"""
    model.eval()
    with torch.no_grad():
        # Prepare source tensor
        if isinstance(source, np.ndarray):
            source = torch.FloatTensor(source)

        # Add batch dimension if needed
        if len(source.shape) == 2:
            source = source.unsqueeze(0)  # [1, seq_len, input_size]

        # Move to device
        source = source.to(device)

        # Get output size (same as input size for this example)
        output_size = source.size(2)

        # Encode
        hidden, cell = model.encoder(source)

        # Start with zeros as first input
        decoder_input = torch.zeros(1, 1, output_size).to(device)
        decoder_input[0, 0, 0] = 1.0  # Set start token (index 0)

        # Store outputs
        outputs = []

        # Generate target sequence
        for _ in range(n_steps):
            decoder_output, hidden, cell = model.decoder(decoder_input, hidden, cell)
            outputs.append(decoder_output.squeeze().detach().cpu().numpy())
            decoder_input = decoder_output

        return np.array(outputs)



In [6]:
# ****************** SECTION 5: MAIN EXECUTION ******************
def run_seq2seq_model():
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Configure problem
    n_features = 50 + 1
    n_steps_in = 6
    n_steps_out = 3
    hidden_size = 256
    batch_size = 128
    num_epochs = 5
    num_layers = 2

    train_dataset = Seq2SeqDataset(n_steps_in, n_steps_out, n_features, 50000)  # More training data
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    encoder = EncoderRNN(n_features, hidden_size, num_layers=num_layers, dropout=0.2).to(device)
    decoder = DecoderRNN(hidden_size, n_features, num_layers=num_layers, dropout=0.2).to(device)
    model = Seq2Seq(encoder, decoder).to(device)

    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2, factor=0.5)

    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for i, batch in enumerate(train_loader):
            source = batch['source'].to(device)
            target = batch['target'].to(device)
            target_in = batch['target_in'].to(device)

            optimizer.zero_grad()

            output = model(source, target_in, teacher_forcing_ratio=0.9 - epoch*0.1)  # Decrease teacher forcing over time

            loss = criterion(output, target)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
            optimizer.step()

            total_loss += loss.item()

            if (i+1) % 100 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

        avg_loss = total_loss/len(train_loader)
        print(f'Epoch [{epoch+1}/{num_epochs}], Average Loss: {avg_loss:.4f}')
        scheduler.step(avg_loss)


    torch.save({
        'encoder_state_dict': encoder.state_dict(),
        'decoder_state_dict': decoder.state_dict(),
        'model_state_dict': model.state_dict(),
    }, 'seq2seq_model.pth')

    model.eval()
    test_dataset = Seq2SeqDataset(n_steps_in, n_steps_out, n_features, 100)
    correct = 0

    for i in range(len(test_dataset)):
        sample = test_dataset[i]
        source = sample['source'].to(device)
        target_raw = sample['target_raw']

        predicted = predict_sequence(model, source, n_steps_out, device)
        predicted_indices = one_hot_decode(predicted)

        if np.array_equal(target_raw, predicted_indices):
            correct += 1

    print(f'Accuracy: {correct/len(test_dataset)*100:.2f}%')

    for i in range(10):
        sample = test_dataset[i]
        source = sample['source'].to(device)
        source_raw = sample['source_raw']
        target_raw = sample['target_raw']

        # Predict
        predicted = predict_sequence(model, source, n_steps_out, device)
        predicted_indices = one_hot_decode(predicted)

        print(f'X={source_raw}, y={target_raw}, yhat={predicted_indices}')

if __name__ == "__main__":
    torch.manual_seed(42)
    np.random.seed(42)
    run_seq2seq_model()

Using device: cuda
Epoch [1/5], Step [100/391], Loss: 0.0695
Epoch [1/5], Step [200/391], Loss: 0.0473
Epoch [1/5], Step [300/391], Loss: 0.0362
Epoch [1/5], Average Loss: 0.0542
Epoch [2/5], Step [100/391], Loss: 0.0162
Epoch [2/5], Step [200/391], Loss: 0.0082
Epoch [2/5], Step [300/391], Loss: 0.0034
Epoch [2/5], Average Loss: 0.0098
Epoch [3/5], Step [100/391], Loss: 0.0011
Epoch [3/5], Step [200/391], Loss: 0.0018
Epoch [3/5], Step [300/391], Loss: 0.0006
Epoch [3/5], Average Loss: 0.0011
Epoch [4/5], Step [100/391], Loss: 0.0004
Epoch [4/5], Step [200/391], Loss: 0.0004
Epoch [4/5], Step [300/391], Loss: 0.0006
Epoch [4/5], Average Loss: 0.0004
Epoch [5/5], Step [100/391], Loss: 0.0004
Epoch [5/5], Step [200/391], Loss: 0.0001
Epoch [5/5], Step [300/391], Loss: 0.0001
Epoch [5/5], Average Loss: 0.0003
Accuracy: 100.00%
X=[24, 30, 17, 21, 28, 26], y=[17, 30, 24], yhat=[np.int64(17), np.int64(30), np.int64(24)]
X=[16, 34, 28, 18, 7, 44], y=[28, 34, 16], yhat=[np.int64(28), np.int64